In [9]:
!pip install --upgrade pip

In [10]:
!pip install torch numpy pandas cartopy matplotlib

  Using cached pandas-2.3.3-cp39-cp39-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (91 kB)
  Using cached Cartopy-0.23.0-cp39-cp39-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (8.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 888.0/888.0 MB 128.8 MB/s  0:00:050:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 122.8 MB/s  0:00:030:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 67.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 148.3 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 24.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 79.0 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 149.4 MB/s  0:00:010:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 141.1 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━

In [11]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

In [12]:
DATASET_DIR = '/cluster/tufts/c26sp1cs0137/data/assignment2_data/dataset'

In [13]:
targets = torch.load(f'{DATASET_DIR}/targets.pt', weights_only=False)

for key in targets.keys():
    print("---------------------------------------------------------------------------")
    print(key)
    print(targets[key])

---------------------------------------------------------------------------
time
['2018-07-13T00:00:00.000000000' '2018-07-13T01:00:00.000000000'
 '2018-07-13T02:00:00.000000000' ... '2021-07-11T21:00:00.000000000'
 '2021-07-11T22:00:00.000000000' '2021-07-11T23:00:00.000000000']
---------------------------------------------------------------------------
variable_names
['TMP@2m_above_ground', 'RH@2m_above_ground', 'UGRD@10m_above_ground', 'VGRD@10m_above_ground', 'GUST@surface', 'APCP_1hr_acc_fcst@surface']
---------------------------------------------------------------------------
values
tensor([[ 2.9425e+02,  5.6812e+01, -2.2344e+00,  1.0342e+00,  5.8047e+00,
          0.0000e+00],
        [ 2.9275e+02,  6.5188e+01, -5.9424e-01,  3.4644e-01,  5.4688e+00,
          0.0000e+00],
        [ 2.9175e+02,  7.0188e+01, -1.5247e-01, -1.0028e-01,  3.9727e+00,
          0.0000e+00],
        ...,
        [ 2.9950e+02,  6.2312e+01,  9.4189e-01,  3.3594e+00,  5.5039e+00,
          0.0000e+00],
   

In [14]:
len(targets["values"])

26280

In [15]:
input1 = torch.load(f'{DATASET_DIR}/inputs/2018/X_2018071300.pt', weights_only=False)



Layer Type,Typical Input Shape,Typical Output Shape,What happens?
Convolution,"(B, 42, 450, 449)","(B, 64, 450, 449)","The ""Channels"" increase (depth), while spatial size stays similar."
MaxPool,"(B, 64, 450, 449)","(B, 64, 225, 224)",The spatial size is cut in half (Height/Width shrink).
Linear (FC),"(B, Flattened_Vector)","(B, 6)","The spatial maps are ""squashed"" into your 6 final target numbers."

In [ ]:
import torch
import torch.nn as nn

# ── 1. Convolution Layer ──────────────────────────────────────────────────────
class ConvLayer(nn.Module):
    def __init__(self, in_channels=42, out_channels=64):
        super().__init__()
        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=3,
            stride=1,
            padding=1   # padding=1 keeps H/W identical with kernel_size=3
        )
        self.bn   = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        # x: (B, 42, 450, 449) → (B, 64, 450, 449)
        return self.relu(self.bn(self.conv(x)))


# ── 2. MaxPool Layer ──────────────────────────────────────────────────────────
class MaxPoolLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        # x: (B, 64, 450, 449) → (B, 64, 225, 224)
        return self.pool(x)

# Linear layer
class FCLayer(nn.Module):
    def __init__(self, num_classes=6):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc      = nn.LazyLinear(num_classes)  # ← infers in_features automatically

    def forward(self, x):
        x = self.flatten(x)
        return self.fc(x)

In [24]:
# ── Quick sanity check ────────────────────────────────────────────────────────
B = 2
x = torch.randn(B, 42, 450, 449)

conv  = ConvLayer()
pool  = MaxPoolLayer()
fc    = FCLayer()

out1 = conv(x)
print("After ConvLayer :", out1.shape)   # (2, 64, 450, 449)

out2 = pool(out1)
print("After MaxPoolLayer:", out2.shape) # (2, 64, 225, 224)

out3 = fc(out2)
print("After FCLayer    :", out3.shape)  # (2, 6)

After ConvLayer : torch.Size([2, 64, 450, 449])
After MaxPoolLayer: torch.Size([2, 64, 225, 224])
After FCLayer    : torch.Size([2, 6])


In [ ]:
# Get the data for a specific year
# Total number of files per year
# 2018: 4128
# 2019: 8760
# 2020: 8784
# 2021: 4608

n_2018 = 4128
n_2019 = 8760
n_2020 = 8784
n_2021 = 4608

start_date_2018 = '2018-07-13T00:00'
start_date_2019 = '2019-01-01T00:00'
start_date_2020 = '2020-01-01T00:00'
start_date_2021 = '2021-01-01T00:00'

def generate_datetime_strings(start_date: str, n: int) -> list[str]:
    """
    Generate YYYYMMDDHH strings for n dates starting from start_date,
    incrementing by 24 hours each step.
    
    Args:
        start_date: Date string in format 'YYYY-MM-DDTHH:MM'
        n: Number of dates to generate
    
    Returns:
        List of strings in YYYYMMDDHH format
    """
    dt = pd.Timestamp(start_date)
    return [("X_" + (dt + pd.Timedelta(hours=1 * i)).strftime('%Y%m%d%H') + ".pt") for i in range(n)]

tensor_names_2018 = generate_datetime_strings(start_date_2018, n_2018)
tensor_names_2019 = generate_datetime_strings(start_date_2019, n_2019)
tensor_names_2020 = generate_datetime_strings(start_date_2020, n_2020)
tensor_names_2021 = generate_datetime_strings(start_date_2021, n_2021)


['X_2018071300.pt',
 'X_2018071301.pt',
 'X_2018071302.pt',
 'X_2018071303.pt',
 'X_2018071304.pt',
 'X_2018071305.pt',
 'X_2018071306.pt',
 'X_2018071307.pt',
 'X_2018071308.pt',
 'X_2018071309.pt',
 'X_2018071310.pt',
 'X_2018071311.pt',
 'X_2018071312.pt',
 'X_2018071313.pt',
 'X_2018071314.pt',
 'X_2018071315.pt',
 'X_2018071316.pt',
 'X_2018071317.pt',
 'X_2018071318.pt',
 'X_2018071319.pt',
 'X_2018071320.pt',
 'X_2018071321.pt',
 'X_2018071322.pt',
 'X_2018071323.pt',
 'X_2018071400.pt',
 'X_2018071401.pt',
 'X_2018071402.pt',
 'X_2018071403.pt',
 'X_2018071404.pt',
 'X_2018071405.pt',
 'X_2018071406.pt',
 'X_2018071407.pt',
 'X_2018071408.pt',
 'X_2018071409.pt',
 'X_2018071410.pt',
 'X_2018071411.pt',
 'X_2018071412.pt',
 'X_2018071413.pt',
 'X_2018071414.pt',
 'X_2018071415.pt',
 'X_2018071416.pt',
 'X_2018071417.pt',
 'X_2018071418.pt',
 'X_2018071419.pt',
 'X_2018071420.pt',
 'X_2018071421.pt',
 'X_2018071422.pt',
 'X_2018071423.pt',
 'X_2018071500.pt',
 'X_2018071501.pt',
